# 00. Revisi?n de datos

Revisi?n inicial de las bases originales. No se eliminan importes negativos ni filas de forma autom?tica.

## Importaci?n e inspecci?n

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display, HTML

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "funciones_reservas.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontr? la ra?z del proyecto con src/funciones_reservas.py.")
sys.path.insert(0, str(ROOT / "src"))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"


In [ ]:
from funciones_reservas import cargar_excel, normalizar_categorias, resumen_calidad

archivo = RAW / "2026 BASE RESERVAS.xlsx"
df1 = cargar_excel(archivo, hoja="EMISION")
df2 = cargar_excel(archivo, hoja="SINIESTRALIDAD")
print(f"Emisi?n: {df1.shape}; Siniestros: {df2.shape}")


In [ ]:
tablas = []
for nombre, df in [("Emisi?n", df1), ("Siniestros", df2)]:
    resumen = resumen_calidad(df)
    tabla = f"""
    <div style="flex:1; min-width:0;">
        <h3>{nombre}</h3>
        <p>Dimensiones: {df.shape[0]} filas ? {df.shape[1]} columnas</p>
        <p>Duplicados completos: {df.duplicated().sum()}</p>
        {resumen.to_html()}
    </div>
    """
    tablas.append(tabla)
display(HTML('<div style="display:flex; gap:30px; align-items:flex-start;">' + ''.join(tablas) + '</div>'))


In [ ]:
# Se conservan las filas: no se aplica dropna() general.
emision = df1.copy()
siniestros = df2.copy()
for nombre, df in [("emision", emision), ("siniestros", siniestros)]:
    columnas = [col for col in ["tipo_embarcacion", "uso"] if col in df]
    if columnas:
        df_actual = normalizar_categorias(df, columnas, minusculas=True)
        df.loc[:, columnas] = df_actual[columnas]
    if "anio_construccion" in df:
        df["anio_construccion"] = pd.to_numeric(df["anio_construccion"], errors="coerce").astype("Int64")
print("Las filas con faltantes se conservan para decidir su tratamiento seg?n cada an?lisis.")


In [ ]:
for nombre, df in [("Emisi?n", emision), ("Siniestros", siniestros)]:
    print(f"--- {nombre} ---")
    resumen = resumen_calidad(df)
    display(resumen[["ceros", "negativos"]])


In [ ]:
for nombre, df in [("Emisi?n", emision), ("Siniestros", siniestros)]:
    print(f"--- {nombre} ---")
    for columna in ["tipo_embarcacion", "uso"]:
        if columna in df:
            print(columna, sorted(df[columna].dropna().astype(str).unique()))


### Presencia de valores negativos

Los importes negativos podrían corresponder a cancelaciones de pólizas, devoluciones o ajustes contables. Lo correcto no seria eliminar estos registros negativos ya que pueden representar algo importante.

Al sumar las primas positivas y negativas obtenemos el monto neto, esto significa que nuestro analisis será en terminos netos monetarios ajustados

Existen tipos de embarcación con primas agregadas negativas en determinados años. 

No hay valores faltantes.

Un cero puede representar la ausencia de un pago.

## Hallazgos

Las decisiones sobre faltantes, signos negativos, categor?as y ceros deben conservarse documentadas antes de calcular reservas.